Import libraries

In [10]:
#import
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np
import torch
from torch.utils.data import TensorDataset, DataLoader
from cfrnet import CFRnet
import sys
from pathlib import Path
project_root = Path("/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking")
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
from utils import seed_everything
from metrics import auuc, auqc, lift, krcc

In [11]:
%time train_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/train_men.csv")
%time test_df =  pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/test_men.csv")
%time val_df = pd.read_csv(r"/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/dataset/Hillstrom/Men/val_men.csv")

CPU times: user 24.6 ms, sys: 2.08 ms, total: 26.7 ms
Wall time: 26.5 ms
CPU times: user 10 ms, sys: 3.88 ms, total: 13.9 ms
Wall time: 13.9 ms
CPU times: user 4.2 ms, sys: 1.94 ms, total: 6.14 ms
Wall time: 6.14 ms


In [12]:
in_features = ['recency', 'history_segment', 'history', 'mens', 'womens',
       'zip_code', 'newbie', 'channel_Multichannel', 'channel_Phone', 'channel_Web']
label_feature = ['conversion']
treatment_feature = ['treatment']

In [13]:
X_train = train_df[in_features].values.astype(float) # type: ignore
y_train = train_df[label_feature].values.astype(float) # type: ignore
t_train = train_df[treatment_feature].values.astype(float) # type: ignore

X_test = test_df[in_features].values.astype(float) # type: ignore
y_test = test_df[label_feature].values.astype(float) # type: ignore
t_test = test_df[treatment_feature].values.astype(float) # type: ignore

X_val = val_df[in_features].values.astype(float) # type: ignore
y_val = val_df[label_feature].values.astype(float) # type: ignore
t_val = val_df[treatment_feature].values.astype(float) # type: ignore

In [14]:
print('X_train[:10]', X_train[:1].astype(float))

X_train[:10] [[-0.21435131  1.6331766   1.0667411   0.90252386 -1.1010233   1.07039981
   1.00043033  2.70003843 -0.88552759 -0.88616046]]


In [15]:
print('y_train[:10]', y_train[:1].astype(float))

y_train[:10] [[0.]]


In [16]:
# Transform to tensor
def to_tensor(arr):
    return torch.tensor(arr, dtype=torch.float32)

x_men_train_t = to_tensor(X_train)
x_men_val_t = to_tensor(X_val)
x_men_test_t = to_tensor(X_test)

y_men_train_t = to_tensor(y_train).reshape(-1, 1)
y_men_val_t = to_tensor(y_val).reshape(-1, 1)
y_men_test_t = to_tensor(y_test).reshape(-1, 1)

# t_train/t_val/t_test cũng tương tự
t_men_train_t = to_tensor(t_train.astype(float)).reshape(-1, 1)
t_men_val_t = to_tensor(t_val.astype(float)).reshape(-1, 1)
t_men_test_t = to_tensor(t_test.astype(float)).reshape(-1, 1)

# Data loader
train_dataset = TensorDataset(x_men_train_t, t_men_train_t, y_men_train_t)
val_dataset = TensorDataset(x_men_val_t, t_men_val_t, y_men_val_t)
test_dataset = TensorDataset(x_men_test_t, t_men_test_t, y_men_test_t)

batch_size = 6400
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0, pin_memory = True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0, pin_memory = True)

print("-------------------------------------------------------------")
print("✅ Completed transform to tensor ✅")
print(f"Shape of train: x={x_men_train_t.shape}; y={y_men_train_t.shape}; t={t_men_train_t.shape}")
print(f"Shape of val: x={x_men_val_t.shape}; y={y_men_val_t.shape}; t={t_men_val_t.shape}")
print(f"Shape of test: x={x_men_test_t.shape}; y={y_men_test_t.shape}; t={t_men_test_t.shape}")

-------------------------------------------------------------
✅ Completed transform to tensor ✅
Shape of train: x=torch.Size([25567, 10]); y=torch.Size([25567, 1]); t=torch.Size([25567, 1])
Shape of val: x=torch.Size([4262, 10]); y=torch.Size([4262, 1]); t=torch.Size([4262, 1])
Shape of test: x=torch.Size([12784, 10]); y=torch.Size([12784, 1]); t=torch.Size([12784, 1])


In [17]:
epochs = 150
alpha = 1
lr = 1e-3
wd = 1e-5
method = "mmd_rbf"
early_stop_metric = "qini"
ema = True
ema_alpha = 0.15
patience = 20
early_stop_start = 0
shared_hidden = 200
outcome_hidden = 100
activation = torch.nn.ReLU
outcome_dropout = 0.0
shared_dropout = 0.0
print (f" epochs = {epochs}")
print (f" method = {method}")
print (f" alpha = {alpha}")
print (f" learning rate = {lr}")
print (f" weight decay = {wd}")
print (f" early stop = {early_stop_metric}")
print (f" use ema = {ema}")
print (f" ema alpha = {ema_alpha}")
print (f" patience = {patience}")
print (f" shared hidden = {shared_hidden}")
print (f" outcome hidden = {outcome_hidden}")
print (f"activation = {activation}")

 epochs = 150
 method = mmd_rbf
 alpha = 1
 learning rate = 0.001
 weight decay = 1e-05
 early stop = qini
 use ema = True
 ema alpha = 0.15
 patience = 20
 shared hidden = 200
 outcome hidden = 100
activation = <class 'torch.nn.modules.activation.ReLU'>


In [18]:
import pandas as pd
import numpy as np
import torch
from itertools import product

# 1. Grid search config
seeds = [412312, 42, 1874, 902745, 1]
lr_grid = [5e-5, 1e-4, 5e-4, 1e-3]
wd_grid = [1e-6, 1e-5, 1e-4]
method_grid = ["mmd_linear", "mmd_rbf"]
alpha_grid = [0.1, 0.5, 1.0]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
grid_results = []

total_cfg = len(lr_grid) * len(wd_grid) * len(method_grid) * len(alpha_grid)
print(f"🔄 Bắt đầu grid search: {total_cfg} cấu hình")

# 2. Loop over all hyperparameter combinations
for grid_lr, grid_wd, grid_method, grid_alpha in product(lr_grid, wd_grid, method_grid, alpha_grid):
    run_rows = []

    print("\n" + "=" * 110)
    print(
        f"Cấu hình: lr={grid_lr:.1e}, weight_decay={grid_wd:.1e}, "
        f"method={grid_method}, alpha={grid_alpha}"
    )
    print("=" * 110)

    for SEED in seeds:
        seed_everything(SEED)

        # Reinitialize model for each seed
        cfr = CFRnet(
            input_dim=x_men_train_t.shape[1],
            epochs=epochs,
            alpha=grid_alpha,
            learning_rate=grid_lr,
            weight_decay=grid_wd,
            method=grid_method,
            use_ema=ema,
            ema_alpha=ema_alpha,
            patience=patience,
            shared_hidden=shared_hidden,
            outcome_hidden=outcome_hidden,
            outcome_dropout=outcome_dropout,
            shared_dropout=shared_dropout,
            early_stop_metric=early_stop_metric,
            early_stop_start_epoch=early_stop_start,
            activation=activation
        )

        cfr.fit(train_loader, val_loader)

        # Validation prediction
        x_val_device = x_men_val_t.to(device)
        y0_val_p, y1_val_p = cfr.predict(x_val_device)
        uplift_val = (y1_val_p - y0_val_p).detach().cpu().numpy().flatten()
        y_val_true_np = y_men_val_t.detach().cpu().numpy().flatten()
        t_val_true_np = t_men_val_t.detach().cpu().numpy().flatten()

        current_val_auqc = auqc(y_val_true_np, t_val_true_np, uplift_val, bins=100, plot=False)

        # Test prediction
        x_test_device = x_men_test_t.to(device)
        y0_pred, y1_pred = cfr.predict(x_test_device)

        uplift_pred = (y1_pred - y0_pred).detach().cpu().numpy().flatten()
        y_true = y_men_test_t.detach().cpu().numpy().flatten()
        t_true = t_men_test_t.detach().cpu().numpy().flatten()

        # ATE error
        ate_pred = uplift_pred.mean()
        ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()

        row = {
            "Seed": SEED,
            "lr": grid_lr,
            "weight_decay": grid_wd,
            "method": grid_method,
            "alpha": grid_alpha,
            "Val_AUQC": current_val_auqc,
            "AUUC": auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
            "AUQC": auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
            "Lift": lift(y_true, t_true, uplift_pred, h=0.3),
            "KRCC": krcc(y_true, t_true, uplift_pred, bins=100),
            "ATE_Err": abs(ate_pred - ate_true)
        }
        run_rows.append(row)

        print(
            f"  ✔ Seed {SEED} | Val_AUQC={current_val_auqc:.4f} | "
            f"Test_AUQC={row['AUQC']:.4f}"
        )

    # Aggregate each hyperparameter combination
    df_pair = pd.DataFrame(run_rows)
    mean_pair = df_pair[["Val_AUQC", "AUUC", "AUQC", "Lift", "KRCC", "ATE_Err"]].mean()
    std_pair = df_pair[["Val_AUQC", "AUUC", "AUQC", "Lift", "KRCC", "ATE_Err"]].std()

    grid_results.append({
        "lr": grid_lr,
        "weight_decay": grid_wd,
        "method": grid_method,
        "alpha": grid_alpha,
        "mean_Val_AUQC": mean_pair["Val_AUQC"],
        "mean_AUUC": mean_pair["AUUC"],
        "mean_AUQC": mean_pair["AUQC"],
        "mean_Lift": mean_pair["Lift"],
        "mean_KRCC": mean_pair["KRCC"],
        "mean_ATE_Err": mean_pair["ATE_Err"],
        "std_Val_AUQC": std_pair["Val_AUQC"],
        "std_AUUC": std_pair["AUUC"],
        "std_AUQC": std_pair["AUQC"],
        "std_Lift": std_pair["Lift"],
        "std_KRCC": std_pair["KRCC"],
        "std_ATE_Err": std_pair["ATE_Err"]
    })

# 3. Final ranking by mean test AUQC
df_grid = pd.DataFrame(grid_results).sort_values("mean_AUQC", ascending=False).reset_index(drop=True)
best_cfg = df_grid.iloc[0]

print("\n" + "=" * 130)
print(f"{'KẾT QUẢ GRID SEARCH (XẾP HẠNG THEO MEAN TEST AUQC)':^130}")
print("=" * 130)

display_cols = [
    "lr", "weight_decay", "method", "alpha",
    "mean_Val_AUQC", "mean_AUUC", "mean_AUQC", "mean_Lift", "mean_KRCC", "mean_ATE_Err",
    "std_Val_AUQC", "std_AUUC", "std_AUQC", "std_Lift", "std_KRCC", "std_ATE_Err"
]

print(df_grid[display_cols].to_string(
    index=False,
    formatters={
        "lr": "{:.1e}".format,
        "weight_decay": "{:.1e}".format,
        "alpha": "{:.1f}".format,
        "mean_Val_AUQC": "{:.4f}".format,
        "mean_AUUC": "{:.4f}".format,
        "mean_AUQC": "{:.4f}".format,
        "mean_Lift": "{:.4f}".format,
        "mean_KRCC": "{:.4f}".format,
        "mean_ATE_Err": "{:.4f}".format,
        "std_Val_AUQC": "{:.4f}".format,
        "std_AUUC": "{:.4f}".format,
        "std_AUQC": "{:.4f}".format,
        "std_Lift": "{:.4f}".format,
        "std_KRCC": "{:.4f}".format,
        "std_ATE_Err": "{:.4f}".format
    }
))

print("-" * 130)
print("Best hyperparameters by mean TEST AUQC:")
print(
    f"  lr={best_cfg['lr']:.1e}, weight_decay={best_cfg['weight_decay']:.1e}, "
    f"method={best_cfg['method']}, alpha={best_cfg['alpha']:.1f}"
)
print(f"  mean TEST AUQC = {best_cfg['mean_AUQC']:.4f} ± {best_cfg['std_AUQC']:.4f}")
print("=" * 130)

🔄 Bắt đầu grid search: 72 cấu hình

Cấu hình: lr=5.0e-05, weight_decay=1.0e-06, method=mmd_linear, alpha=0.1
Locked random seed: 412312
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 1
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/150 | Loss: 0.5111 | Val Loss: 0.5102 | Val Qini: 0.4004 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4004 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.5073 | Val Loss: 0.5068 | Val Qini: 0.4184 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4031 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.5040 | Val Loss: 0.5035 | Val Qini: 0.4405 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4087 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.5009 | Val Loss: 0.5004 | Val Qini: 0.4471 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.4145 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.4979 | Val Loss: 0.4974 | Val Qini: 0.4

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.5237 | Val Loss: 0.5236 | Val Qini: 0.3384 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3384 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.5220 | Val Loss: 0.5205 | Val Qini: 0.3475 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3398 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.5190 | Val Loss: 0.5175 | Val Qini: 0.3583 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3426 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.5159 | Val Loss: 0.5144 | Val Qini: 0.3737 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3472 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.5124 | Val Loss: 0.5113 | Val Qini: 0.3865 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3531 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 0.5085 | Val Loss: 0.5080 | Val Qini: 0.4039 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3607 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 0.5052 | Val Loss: 0.5045 | Val Qini: 0.4137 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3687 | ⭐ NEW BEST (peak ≥ trend)
Epoch 8/150 | Loss: 0.5009 | Val Loss: 0.5008 | 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.5023 | Val Loss: 0.5019 | Val Qini: 0.3041 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3041 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.4994 | Val Loss: 0.4985 | Val Qini: 0.3070 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3046 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.4945 | Val Loss: 0.4951 | Val Qini: 0.3083 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3051 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.4914 | Val Loss: 0.4916 | Val Qini: 0.3078 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.3055 | ✓ above trend but not peak (patience: 1/20)
Epoch 5/150 | Loss: 0.4901 | Val Loss: 0.4879 | Val Qini: 0.3057 ✓ above trend but not peak (patience: 2/20)EMA Trend: 0.3055 | ✓ above trend but not peak (patience: 2/20)
Epoch 6/150 | Loss: 0.4846 | Val Loss: 0.4841 | Val Qini: 0.3145 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3069 | ⭐ NEW BEST (peak ≥ trend)
Epoch 7/150 | Loss: 0.4804 | Val Loss: 0.4801 | Val Qini: 0.3427 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3123 | ⭐ 

...

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: -7.5533 | Val Loss: -7.5875 | Val Qini: 0.7277 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7277 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -7.7511 | Val Loss: -7.8217 | Val Qini: 0.7267 (patience: 1/20)EMA Trend: 0.7275 | (patience: 1/20)
Epoch 3/150 | Loss: -7.9656 | Val Loss: -7.9790 | Val Qini: 0.2766 (patience: 2/20)EMA Trend: 0.6599 | (patience: 2/20)
Epoch 4/150 | Loss: -7.9830 | Val Loss: -7.9818 | Val Qini: 0.2852 (patience: 3/20)EMA Trend: 0.6037 | (patience: 3/20)
Epoch 5/150 | Loss: -7.9830 | Val Loss: -7.9817 | Val Qini: 0.2888 (patience: 4/20)EMA Trend: 0.5565 | (patience: 4/20)
Epoch 6/150 | Loss: -7.9845 | Val Loss: -7.9817 | Val Qini: 0.2908 (patience: 5/20)EMA Trend: 0.5166 | (patience: 5/20)
Epoch 7/150 | Loss: -7.9797 | Val Loss: -7.9817 | Val Qini: 0.2949 (patience: 6/20)EMA Trend: 0.4834 | (patience: 6/20)
Epoch 8/150 | Loss: -7.9814 | Val Loss: -7.9817 | Val Qini: 0.2960 (patience: 7/20)EMA Trend: 0.4553 | (patience: 7/20)
Epoch 9/150 | Loss: -7

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: -7.5385 | Val Loss: -7.5633 | Val Qini: 0.6587 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.6587 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: -7.6915 | Val Loss: -7.7569 | Val Qini: 0.2457 (patience: 1/20)EMA Trend: 0.5968 | (patience: 1/20)
Epoch 3/150 | Loss: -7.9487 | Val Loss: -7.9717 | Val Qini: 0.5532 (patience: 2/20)EMA Trend: 0.5902 | (patience: 2/20)
Epoch 4/150 | Loss: -7.9796 | Val Loss: -7.9819 | Val Qini: 0.7355 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.6120 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: -7.9812 | Val Loss: -7.9818 | Val Qini: 0.7385 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.6310 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: -7.9769 | Val Loss: -7.9818 | Val Qini: 0.7350 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.6466 | ✓ above trend but not peak (patience: 1/20)
Epoch 7/150 | Loss: -7.9800 | Val Loss: -7.9818 | Val Qini: 0.7278 ✓ above trend but not peak (patience: 2/20)EMA Trend: 0.6588 | ✓ above trend but not peak (patience: 2/20

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)
/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


In [22]:
import pandas as pd
import numpy as np
import torch

# 1. Cấu hình
seeds = [412312, 42, 1874, 902745, 1]
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
all_runs = []

print(f"🔄 Đang huấn luyện TARNet trên {len(seeds)} seeds khác nhau... Vui lòng đợi.")

# 2. Vòng lặp chạy (Chỉ xử lý, không in kết quả lẻ)
for SEED in seeds:
    seed_everything(SEED)
    
    # Khởi tạo lại mô hình
    cfrnet = CFRnet(
        input_dim=x_men_train_t.shape[1], epochs=epochs, learning_rate=0.00005, 
        alpha=1, method="mmd_linear",
        weight_decay=0.000001, use_ema=ema, ema_alpha=ema_alpha, patience=patience,
        shared_hidden=shared_hidden, outcome_hidden=outcome_hidden,
        early_stop_metric=early_stop_metric, early_stop_start_epoch=early_stop_start,
        activation=activation
    )
    
    cfrnet.fit(train_loader, val_loader)
    
    # Predict
    x_men_test_t_on_device = x_men_test_t.to(device)
    y0_pred, y1_pred = cfrnet.predict(x_men_test_t_on_device)
    
    uplift_pred = (y1_pred - y0_pred).cpu().numpy().flatten()
    y_true = y_men_test_t.cpu().numpy().flatten()
    t_true = t_men_test_t.cpu().numpy().flatten()
    
    # Tính toán
    ate_pred = uplift_pred.mean()
    ate_true = y_true[t_true == 1].mean() - y_true[t_true == 0].mean()
    
    all_runs.append({
        'Seed': SEED,
        'AUUC': auuc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'AUQC': auqc(y_true, t_true, uplift_pred, bins=100, plot=False),
        'Lift': lift(y_true, t_true, uplift_pred, h=0.3),
        'KRCC': krcc(y_true, t_true, uplift_pred, bins=100),
        'ATE_Err': abs(ate_pred - ate_true)
    })
    print(f"  ✔️ Đã xong Seed {SEED}")

# 3. HIỂN THỊ KẾT QUẢ TỔNG HỢP (Print 1 lúc)
df_results = pd.DataFrame(all_runs)

print("\n" + "="*85)
print(f"{'CHI TIẾT TỪNG SEED':^85}")
print("="*85)
# Sử dụng to_string để in bảng đẹp mắt
print(df_results.to_string(index=False, formatters={
    'AUUC': '{:,.4f}'.format, 'AUQC': '{:,.4f}'.format, 
    'Lift': '{:,.4f}'.format, 'KRCC': '{:,.4f}'.format, 'ATE_Err': '{:,.4f}'.format
}))

# 4. Tính toán Mean và Std
mean_res = df_results.drop(columns='Seed').mean()
std_res = df_results.drop(columns='Seed').std()

print("="*85)
print(f"{'KẾT QUẢ TRUNG BÌNH (MEAN ± STD)':^85}")
print("-" * 85)
summary_data = []
for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
    print(f"{metric:<10}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")

print("="*85)

🔄 Đang huấn luyện TARNet trên 5 seeds khác nhau... Vui lòng đợi.
Locked random seed: 412312
🔃🔃🔃Begin training Dragonnet🔃🔃🔃
📊 Early Stop Metric: QINI
📊 Early Stop Start Epoch: 1
📊 Strategy: Two-Stage EMA Filter (alpha=0.15)
   EMA filters noise spikes, Raw Qini determines peak height
   Select checkpoint: raw_qini is highest AND raw_qini >= ema_qini
Epoch 1/150 | Loss: 0.5420 | Val Loss: 0.5402 | Val Qini: 0.3910 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3910 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.5329 | Val Loss: 0.5369 | Val Qini: 0.3984 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3921 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.5300 | Val Loss: 0.5338 | Val Qini: 0.4075 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3944 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.5299 | Val Loss: 0.5308 | Val Qini: 0.4117 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3970 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.5241 | Val Loss: 0.5278 | Val Qini: 0.4064 ✓ above trend but not peak (patience: 1/

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.5461 | Val Loss: 0.5525 | Val Qini: 0.3413 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3413 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.5578 | Val Loss: 0.5497 | Val Qini: 0.3519 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3429 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.5538 | Val Loss: 0.5471 | Val Qini: 0.3590 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3453 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.5495 | Val Loss: 0.5446 | Val Qini: 0.3591 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3474 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.5441 | Val Loss: 0.5421 | Val Qini: 0.3585 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.3491 | ✓ above trend but not peak (patience: 1/20)
Epoch 6/150 | Loss: 0.5344 | Val Loss: 0.5395 | Val Qini: 0.3570 ✓ above trend but not peak (patience: 2/20)EMA Trend: 0.3502 | ✓ above trend but not peak (patience: 2/20)
Epoch 7/150 | Loss: 0.5339 | Val Loss: 0.5370 | Val Qini: 0.3614 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3519 | ⭐ 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.5293 | Val Loss: 0.5332 | Val Qini: 0.3031 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3031 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.5298 | Val Loss: 0.5302 | Val Qini: 0.3076 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3038 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.5136 | Val Loss: 0.5273 | Val Qini: 0.3120 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3050 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.5168 | Val Loss: 0.5244 | Val Qini: 0.3122 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3061 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.5320 | Val Loss: 0.5214 | Val Qini: 0.3088 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.3065 | ✓ above trend but not peak (patience: 1/20)
Epoch 6/150 | Loss: 0.5147 | Val Loss: 0.5183 | Val Qini: 0.3067 ✓ above trend but not peak (patience: 2/20)EMA Trend: 0.3066 | ✓ above trend but not peak (patience: 2/20)
Epoch 7/150 | Loss: 0.5066 | Val Loss: 0.5152 | Val Qini: 0.3177 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.3082 | ⭐ 

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.5517 | Val Loss: 0.5440 | Val Qini: 0.7241 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7241 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.5466 | Val Loss: 0.5403 | Val Qini: 0.7533 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7285 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.5409 | Val Loss: 0.5365 | Val Qini: 0.7692 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7346 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.5390 | Val Loss: 0.5329 | Val Qini: 0.7731 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7404 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.5377 | Val Loss: 0.5292 | Val Qini: 0.7768 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.7458 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 0.5281 | Val Loss: 0.5255 | Val Qini: 0.7747 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.7502 | ✓ above trend but not peak (patience: 1/20)
Epoch 7/150 | Loss: 0.5193 | Val Loss: 0.5218 | Val Qini: 0.7722 ✓ above trend but not peak (patience: 2/20)EMA Trend: 0.7535 | ✓ above trend but no

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


Epoch 1/150 | Loss: 0.5359 | Val Loss: 0.5391 | Val Qini: 0.5394 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5394 | ⭐ NEW BEST (peak ≥ trend)
Epoch 2/150 | Loss: 0.5360 | Val Loss: 0.5366 | Val Qini: 0.5428 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5399 | ⭐ NEW BEST (peak ≥ trend)
Epoch 3/150 | Loss: 0.5340 | Val Loss: 0.5341 | Val Qini: 0.5492 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5413 | ⭐ NEW BEST (peak ≥ trend)
Epoch 4/150 | Loss: 0.5325 | Val Loss: 0.5316 | Val Qini: 0.5509 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5427 | ⭐ NEW BEST (peak ≥ trend)
Epoch 5/150 | Loss: 0.5257 | Val Loss: 0.5291 | Val Qini: 0.5541 ⭐ NEW BEST (peak ≥ trend)EMA Trend: 0.5444 | ⭐ NEW BEST (peak ≥ trend)
Epoch 6/150 | Loss: 0.5313 | Val Loss: 0.5268 | Val Qini: 0.5531 ✓ above trend but not peak (patience: 1/20)EMA Trend: 0.5457 | ✓ above trend but not peak (patience: 1/20)
Epoch 7/150 | Loss: 0.5231 | Val Loss: 0.5244 | Val Qini: 0.5532 ✓ above trend but not peak (patience: 2/20)EMA Trend: 0.5469 | ✓ above trend but no

/home/ducvu0904/Documents/Lab/Conversion vs revenue benchmarking/Conversion/CFRnet/cfrnet.py:272: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  x = torch.tensor(x, dtype=torch.float32, device=self.device)


In [21]:
# Read existing results from memory (no retraining)
print('=== BEST CONFIG (from grid search) ===')
if 'best_cfg' in globals():
    print(best_cfg[['lr', 'weight_decay', 'method', 'alpha', 'mean_Val_AUQC', 'mean_AUQC', 'std_AUQC']].to_string())
else:
    print('best_cfg not found in current kernel state')

print('\n=== TOP 10 GRID ROWS ===')
if 'df_grid' in globals():
    cols = ['lr', 'weight_decay', 'method', 'alpha', 'mean_Val_AUQC', 'mean_AUQC', 'std_AUQC']
    print(df_grid[cols].head(10).to_string(index=False))
else:
    print('df_grid not found in current kernel state')

print('\n=== PER-SEED SUMMARY ===')
if 'df_results' in globals():
    print(df_results.to_string(index=False))
    print('\nMean ± Std:')
    mean_res = df_results.drop(columns='Seed').mean()
    std_res = df_results.drop(columns='Seed').std()
    for metric in ['AUUC', 'AUQC', 'Lift', 'KRCC', 'ATE_Err']:
        print(f"{metric}: {mean_res[metric]:.4f} ± {std_res[metric]:.4f}")
else:
    print('df_results not found in current kernel state')

=== BEST CONFIG (from grid search) ===
lr                  0.00005
weight_decay       0.000001
method           mmd_linear
alpha                   1.0
mean_Val_AUQC      0.623299
mean_AUQC          0.566372
std_AUQC           0.092193

=== TOP 10 GRID ROWS ===
     lr  weight_decay     method  alpha  mean_Val_AUQC  mean_AUQC  std_AUQC
0.00005      0.000001 mmd_linear    1.0       0.623299   0.566372  0.092193
0.00005      0.000010 mmd_linear    1.0       0.618940   0.564365  0.089781
0.00050      0.000001    mmd_rbf    1.0       0.744115   0.561918  0.038013
0.00050      0.000001    mmd_rbf    0.1       0.743219   0.560753  0.038051
0.00050      0.000001    mmd_rbf    0.5       0.744896   0.560636  0.037760
0.00100      0.000001    mmd_rbf    0.5       0.748212   0.558931  0.027514
0.00100      0.000010    mmd_rbf    1.0       0.749424   0.558419  0.028177
0.00100      0.000010    mmd_rbf    0.1       0.748836   0.558202  0.028836
0.00100      0.000001    mmd_rbf    0.1       0.748267 